# Iyer retrain στο native 8 kHz

Παλιά κάναμε resample σε 44.1 kHz στο preprocessing. Αυτό δημιουργούσε bandwidth mismatch: τα Iyer audio είναι έως 4 kHz (bandlimited από τηλεφωνία), αλλά οι MFCCs υπολογίζονταν με 22 kHz Nyquist.

Τώρα κρατάμε native 8 kHz και ξανατρέφουμε.

In [1]:
import sys, joblib, tempfile
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr
from pathlib import Path
from tqdm import tqdm

from src.features import extract_features, FEATURE_NAMES

MODELS = Path('../models')
DATA = Path('../data/iyer')

NATIVE_SR = 8000  # Iyer native sample rate

In [2]:
def preprocess_at_native_sr(wav_path, target_sr=NATIVE_SR):
    """Preprocess κρατώντας native sample rate (όχι upsampling)."""
    y, sr = librosa.load(str(wav_path), sr=target_sr, mono=True)
    
    # Noise reduction
    intervals = librosa.effects.split(y, top_db=20, frame_length=1024, hop_length=256)
    silent_mask = np.ones(len(y), dtype=bool)
    for start, end in intervals:
        silent_mask[start:end] = False
    noise_sample = y[silent_mask]
    
    try:
        if len(noise_sample) > sr * 0.1:
            y = nr.reduce_noise(y=y, sr=sr, y_noise=noise_sample, stationary=True, prop_decrease=0.7)
        else:
            y = nr.reduce_noise(y=y, sr=sr, stationary=False, prop_decrease=0.5)
    except Exception:
        pass
    
    # Trim + normalize
    y, _ = librosa.effects.trim(y, top_db=25)
    rms = np.sqrt(np.mean(y**2))
    if rms > 1e-6:
        y = y * (10**(-25/20) / rms)
    peak = np.max(np.abs(y))
    if peak > 1.0:
        y = y / peak
    
    return y, sr

In [3]:
# Re-extract Iyer features at 8 kHz native
rows = []
for cls, label, folder in [('HC', 0, 'HC_AH'), ('PD', 1, 'PD_AH')]:
    files = list((DATA / folder).glob('*.wav'))
    for wav in tqdm(files, desc=f'Iyer {cls}'):
        try:
            y, sr = preprocess_at_native_sr(wav)
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                sf.write(tmp.name, y, sr)
                feats = extract_features(tmp.name)
            feats['class'] = label
            feats['subject'] = wav.stem
            feats['filename'] = wav.name
            rows.append(feats)
        except Exception as e:
            print(f'Error {wav.name}: {e}')

iyer_df = pd.DataFrame(rows)
iyer_df.to_csv(DATA / 'iyer_features_8khz.csv', index=False)
print(f'\n{iyer_df.shape[0]} samples extracted at 8 kHz native')
print(f'Classes: {iyer_df["class"].value_counts().to_dict()}')

Iyer HC:   0%|          | 0/41 [00:00<?, ?it/s]

Iyer HC:   2%|▏         | 1/41 [00:00<00:32,  1.22it/s]

Iyer HC:  12%|█▏        | 5/41 [00:00<00:05,  6.85it/s]

Iyer HC:  22%|██▏       | 9/41 [00:01<00:02, 12.10it/s]

Iyer HC:  29%|██▉       | 12/41 [00:01<00:01, 15.51it/s]

...

Iyer PD:  80%|████████  | 32/40 [00:01<00:00, 28.23it/s]

Iyer PD:  88%|████████▊ | 35/40 [00:01<00:00, 27.36it/s]

Iyer PD:  95%|█████████▌| 38/40 [00:01<00:00, 26.99it/s]

Iyer PD: 100%|██████████| 40/40 [00:01<00:00, 27.24it/s]


81 samples extracted at 8 kHz native
Classes: {0: 41, 1: 40}


In [4]:
# Compare key features 8 kHz vs 44.1 kHz
iyer_old = pd.read_csv(DATA / 'iyer_features.csv')

print(f'{"Feature":<25s} {"44.1 kHz (mean ± std)":<30s} {"8 kHz native (mean ± std)":<30s}')
print('-' * 90)
for f in ['locPctJitter', 'locShimmer', 'meanHarmToNoiseHarmonicity',
          'std_MFCC_0th_coef', 'std_MFCC_1st_coef', 'std_MFCC_2nd_coef',
          'mean_MFCC_0th_coef', 'mean_MFCC_6th_coef']:
    old_m = iyer_old[f].mean()
    old_s = iyer_old[f].std()
    new_m = iyer_df[f].mean()
    new_s = iyer_df[f].std()
    print(f'{f:<25s} {old_m:>10.3f} ± {old_s:>8.3f}     {new_m:>10.3f} ± {new_s:>8.3f}')

Feature                   44.1 kHz (mean ± std)          8 kHz native (mean ± std)     
------------------------------------------------------------------------------------------
locPctJitter                   0.006 ±    0.004          0.006 ±    0.003
locShimmer                     0.070 ±    0.030          0.070 ±    0.031
meanHarmToNoiseHarmonicity      8.676 ±    6.467          8.337 ±    6.312
std_MFCC_0th_coef             31.520 ±   16.255         53.891 ±   31.478
std_MFCC_1st_coef             23.921 ±   12.854         14.996 ±    6.827
std_MFCC_2nd_coef             13.637 ±    5.603         13.281 ±    6.515
mean_MFCC_0th_coef          -403.537 ±   24.305       -219.533 ±   38.770
mean_MFCC_6th_coef           -32.804 ±   13.053        -15.491 ±    9.948


## Train new Iyer model at 8 kHz

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

X = iyer_df[FEATURE_NAMES]
y = iyer_df['class'].values
groups = iyer_df['subject'].values

pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')),
])

y_proba = cross_val_predict(pipe, X, y, cv=GroupKFold(5), groups=groups, method='predict_proba')[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(f'Iyer at 8 kHz: acc={accuracy_score(y, y_pred):.3f}, F1={f1_score(y, y_pred):.3f}, MCC={matthews_corrcoef(y, y_pred):.3f}')
cm = confusion_matrix(y, y_pred)
print(f'CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

pipe.fit(X, y)
joblib.dump(pipe, MODELS / 'iyer_8khz.joblib')
print('\nSaved iyer_8khz.joblib')

Iyer at 8 kHz: acc=0.728, F1=0.725, MCC=0.457
CM: TN=30, FP=11, FN=11, TP=29

Saved iyer_8khz.joblib
